# Anomaly Detection - Manual Exploration

This notebook demonstrates different methods for anomaly detection for static code analysis data using jQAssistant and Neo4j. It shows different approaches from simple statistical methods to more complex machine learning algorithms. The focus is on detecting anomalies in the data, which can be useful for identifying potential issues or areas for improvement in the codebase.

<br>  

### References
- [jqassistant](https://jqassistant.org)
- [Neo4j Python Driver](https://neo4j.com/docs/api/python-driver/current)

In [ ]:
import os
import typing

from IPython.display import display
import pandas as pd
import numpy as np

import matplotlib.pyplot as plot
import seaborn

In [ ]:
#The following cell uses the build-in %html "magic" to override the CSS style for tables to a much smaller size.
#This is especially needed for PDF export of tables with multiple columns.

In [ ]:
%%html
<style>
/* CSS style for smaller dataframe tables. */
.dataframe th {
    font-size: 8px;
}
.dataframe td {
    font-size: 8px;
}
</style>

In [ ]:
# Main Colormap
# main_color_map = 'nipy_spectral'
main_color_map = 'viridis'

In [ ]:
from sys import version as python_version
print('Python version: {}'.format(python_version))

from numpy import __version__ as numpy_version
print('numpy version: {}'.format(numpy_version))

from pandas import __version__ as pandas_version
print('pandas version: {}'.format(pandas_version))

from matplotlib import __version__ as matplotlib_version
print('matplotlib version: {}'.format(matplotlib_version))

from seaborn import __version__ as seaborn_version  # type: ignore
print('seaborn version: {}'.format(seaborn_version))

from neo4j import __version__ as neo4j_version
print('neo4j version: {}'.format(neo4j_version))

In [ ]:
# Please set the environment variable "NEO4J_INITIAL_PASSWORD" in your shell 
# before starting jupyter notebook to provide the password for the user "neo4j". 
# It is not recommended to hardcode the password into jupyter notebook for security reasons.
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    uri="bolt://localhost:7687", 
    auth=("neo4j", os.environ.get("NEO4J_INITIAL_PASSWORD"))
)
driver.verify_connectivity()

In [ ]:
def query_cypher_to_data_frame(query: typing.LiteralString, parameters: typing.Optional[typing.Dict[str, typing.Any]] = None):
    records, summary, keys = driver.execute_query(query, parameters_=parameters)
    return pd.DataFrame([record.values() for record in records], columns=keys)

In [ ]:
def get_clustering_property_name(clustering_property_type: typing.Literal['Label', 'Probability'] = 'Label', clustering_name: str = "TunedHDBSCAN"):
    """
    Assembles the property name for clustering results.
    This helps to have a uniform schema.
    """
    return 'clustering' + clustering_name + clustering_property_type

def plot_2d_node_embeddings(
    node_embeddings_for_visualization: pd.DataFrame,
    title: str,
    clustering_name: str = "TunedHDBSCAN",
    main_color_map: str = "tab20",
    x_position_column = 'embeddingVisualizationX',
    y_position_column = 'embeddingVisualizationY'
) -> None:
    if node_embeddings_for_visualization.empty:
        print("No projected data to plot available")
        return

    # Create figure and subplots
    figure, (leiden_subplot, hdbscan_subplot) = plot.subplots(nrows=2, ncols=1, figsize=(10, 12))
    figure.suptitle(title)
    figure.subplots_adjust(top=0.94, left=0.05, right=0.95, bottom=0.04, hspace=0.25)

    # Setup columns
    cluster_label_column_name = get_clustering_property_name('Label', clustering_name)
    node_size_column = 'centrality'

    # Separate HDBSCAN non-noise and noise nodes
    node_embeddings_without_noise = node_embeddings_for_visualization[node_embeddings_for_visualization[cluster_label_column_name] != -1]
    node_embeddings_noise_only = node_embeddings_for_visualization[node_embeddings_for_visualization[cluster_label_column_name] == -1]

    # ------------------------------------------
    # Top subplot: Leiden Communities with KDE
    # ------------------------------------------
    leiden_subplot.set_title("Leiden Community Detection")

    unique_community_ids = node_embeddings_for_visualization["communityId"].unique()
    leiden_color_palette = seaborn.color_palette(main_color_map, len(unique_community_ids))
    leiden_community_to_color = dict(zip(unique_community_ids, leiden_color_palette))

    for community_id in unique_community_ids:
        community_nodes = node_embeddings_for_visualization[
            node_embeddings_for_visualization["communityId"] == community_id
        ]

        # KDE cloud shape
        seaborn.kdeplot(
            x=community_nodes[x_position_column],
            y=community_nodes[y_position_column],
            fill=True,
            alpha=0.12,
            levels=3,
            color=leiden_community_to_color[community_id],
            ax=leiden_subplot,
        )

        # Node scatter points
        leiden_subplot.scatter(
            x=community_nodes[x_position_column],
            y=community_nodes[y_position_column],
            s=community_nodes[node_size_column] * 300,
            color=leiden_community_to_color[community_id],
            alpha=0.7,
            label=f"Community {community_id}"
        )

    leiden_subplot.legend(title="Leiden Communities", loc="best", prop={'size': 6})

    # ------------------------------------------
    # Bottom subplot: HDBSCAN Clustering with KDE
    # ------------------------------------------
    hdbscan_subplot.set_title("HDBSCAN Clustering")

    unique_cluster_labels = node_embeddings_without_noise[cluster_label_column_name].unique()
    hdbscan_color_palette = seaborn.color_palette(main_color_map, len(unique_cluster_labels))
    hdbscan_cluster_to_color = dict(zip(unique_cluster_labels, hdbscan_color_palette))

    for cluster_label in unique_cluster_labels:
        cluster_nodes = node_embeddings_without_noise[
            node_embeddings_without_noise[cluster_label_column_name] == cluster_label
        ]

        # KDE cloud shape
        seaborn.kdeplot(
            x=cluster_nodes[x_position_column],
            y=cluster_nodes[y_position_column],
            fill=True,
            alpha=0.05,
            levels=2,
            color=hdbscan_cluster_to_color[cluster_label],
            ax=hdbscan_subplot,
            # linewidths=0
        )

        # Node scatter points
        hdbscan_subplot.scatter(
            x=cluster_nodes[x_position_column],
            y=cluster_nodes[y_position_column],
            s=cluster_nodes[node_size_column] * 300,
            color=hdbscan_cluster_to_color[cluster_label],
            alpha=0.9,
            label=f"Cluster {cluster_label}"
        )

    # Plot noise points in gray
    hdbscan_subplot.scatter(
        x=node_embeddings_noise_only[x_position_column],
        y=node_embeddings_noise_only[y_position_column],
        s=node_embeddings_noise_only[node_size_column] * 300,
        color='lightgrey',
        alpha=0.4,
        label="Noise"
    )

    hdbscan_subplot.legend(title="HDBSCAN Clusters", loc="best", prop={'size': 6})


## 1. Java Packages

### 1.1 Differences between Page Rank and Article Rank

A high difference between Page Rank and Article Rank can reveal nodes with imbalanced roles — e.g. utility code that is highly depended on but does not depend on much else.

PageRank measures how important a node is by who depends on it (high in-degree weight) while ArticleRank measures how important a node is based on how many other nodes it links to (outgoing edges matter more).

Nodes with low PageRank but high ArticleRank may be coordination-heavy, which could signal:
- Unusual architecture
- Utility overuse
- Monolithic patterns

These are often design smells or potential anomalies in large-scale codebases.

In [ ]:
java_package_centrality_features_query = """
    MATCH (artifact:Java:Artifact)-[:CONTAINS]->(codeUnit:Java:Package)
    WHERE codeUnit.incomingDependencies                        IS NOT NULL
      AND codeUnit.outgoingDependencies                        IS NOT NULL
      AND codeUnit.centralityArticleRank                       IS NOT NULL
      AND codeUnit.centralityPageRank                          IS NOT NULL
      AND codeUnit.centralityBetweenness                       IS NOT NULL
   RETURN DISTINCT 
         codeUnit.fqn                                         AS codeUnitName
        ,codeUnit.name                                        AS shortCodeUnitName
        ,artifact.name                                        AS projectName
        ,codeUnit.incomingDependencies                        AS incomingDependencies
        ,codeUnit.outgoingDependencies                        AS outgoingDependencies
        ,codeUnit.centralityArticleRank                       AS articleRank
        ,codeUnit.centralityPageRank                          AS pageRank
        ,codeUnit.centralityBetweenness                       AS betweenness
"""

java_package_centrality_features = query_cypher_to_data_frame(java_package_centrality_features_query)
display(java_package_centrality_features.head(5))

In [ ]:
def plot_difference_between_article_and_page_rank(
    page_ranks: pd.Series, 
    article_ranks: pd.Series,
    short_names: pd.Series,
) -> None:
    """
    Plots the difference between Article Rank and Page Rank for Java packages.
    
    Parameters
    ----------
    page_ranks : pd.Series
        DataFrame column containing Page Rank values.
    article_ranks : pd.Series
        DataFrame column containing Article Rank values.
    short_names : pd.Series
        DataFrame column containing short names of the code units.
    """
    if page_ranks.empty or article_ranks.empty or short_names.empty:
        print("No data available to plot.")
        return

    # Calculate the difference between Article Rank and Page Rank
    page_to_article_rank_difference = page_ranks - article_ranks
    mean_difference = page_to_article_rank_difference.mean()
    standard_deviation = page_to_article_rank_difference.std()

    plot.figure(figsize=(10, 6))
    plot.hist(page_to_article_rank_difference, bins=50, color='blue', alpha=0.7, edgecolor='black')
    plot.title('Distribution of Page Rank - Article Rank Difference')
    plot.xlabel('Absolute difference between Page Rank and Article Rank')
    plot.ylabel('Frequency')
    plot.xlim(left=page_to_article_rank_difference.min(), right=page_to_article_rank_difference.max())
    plot.yscale('log')  # Use logarithmic scale for better visibility of differences
    plot.grid(True)
    plot.tight_layout()

    # Vertical line for the mean
    plot.axvline(mean_difference, color='red', linestyle='dashed', linewidth=1, label=f'Mean: {mean_difference:.2f}')
    plot.legend()

    # Vertical line for the standard deviation + mean (=z-score of 1)
    positive_z_score_1 = mean_difference + standard_deviation
    negative_z_score_1 = mean_difference - standard_deviation
    plot.axvline(positive_z_score_1, color='orange', linestyle='dashed', linewidth=1, label=f'Mean + 1 x Standard Deviation (z-score 1): {positive_z_score_1:.2f}')
    plot.axvline(negative_z_score_1, color='orange', linestyle='dashed', linewidth=1)
    plot.legend()

    # Vertical line for 2 x standard deviations + mean (=z-score of 2)
    positive_z_score_2 = mean_difference + 2 * standard_deviation
    negative_z_score_2 = mean_difference - 2 * standard_deviation
    plot.axvline(positive_z_score_2, color='green', linestyle='dashed', linewidth=1, label=f'Mean + 2 x Standard Deviation (z-score 2): {2 * positive_z_score_2:.2f}')
    plot.axvline(negative_z_score_2, color='green', linestyle='dashed', linewidth=1)
    plot.legend()

    def annotate_outliers(outliers: pd.DataFrame) -> None:
        if outliers.empty:
            return
        for index, row in outliers.iterrows():
            short_name = row['shortName']
            value = row['pageToArticleRankDifference']
            rank_of_page_rank = row['page_rank_ranking']
            rank_of_article_rank = row['article_rank_ranking']
            x_index_offset = - index * 10 if value > 0 else + index * 10
            plot.annotate(
                f'{short_name} (rank #{rank_of_page_rank}, #{rank_of_article_rank})',
                xy=(value, 1),
                xytext=(value + x_index_offset, 60),
                textcoords='offset points',
                arrowprops=dict(arrowstyle='->', color='black', alpha=0.4),
                fontsize=6,
                rotation=90,
                backgroundcolor='white',
                bbox=dict(boxstyle='round,pad=0.4',
                          edgecolor='silver',
                          facecolor='whitesmoke', alpha=1)
            )

    # Merge all series into a single DataFrame for easier handling
    page_to_article_rank_dataframe = pd.DataFrame({
        'shortName': short_names,
        'pageRank': page_ranks,
        'articleRank': article_ranks,
        'pageToArticleRankDifference': page_to_article_rank_difference,
        'page_rank_ranking': page_ranks.rank().astype(int),
        'article_rank_ranking': article_ranks.rank().astype(int)
    }, index=page_ranks.index)

    # Annotate values above z-score of 2 with their names
    positive_outliers = page_to_article_rank_dataframe[page_to_article_rank_difference > positive_z_score_2].sort_values(by='pageToArticleRankDifference', ascending=False).reset_index().head(5)
    annotate_outliers(positive_outliers)

    # Annotate values below z-score of -2 with their names
    negative_outliers = page_to_article_rank_dataframe[page_to_article_rank_difference < negative_z_score_2].sort_values(by='pageToArticleRankDifference', ascending=True).reset_index().head(5)
    annotate_outliers(negative_outliers)

    plot.show()

In [ ]:
plot_difference_between_article_and_page_rank(
    java_package_centrality_features['pageRank'],
    java_package_centrality_features['articleRank'],
    java_package_centrality_features['shortCodeUnitName']
)

### 1.2 Local Clustering Coefficient

The local clustering coefficient is a measure of how connected a node's neighbors are to each other.
A high local clustering coefficient indicates that a node's neighbors are well-connected, which can suggest a tightly-knit group of related components or classes.
A low local clustering coefficient may indicate that a node's neighbors are not well-connected, which can suggest a more loosely-coupled architecture or potential design smells.

In [ ]:
java_package_clustering_coefficient_query = """
    MATCH (artifact:Java:Artifact)-[:CONTAINS]->(codeUnit:Java:Package)
    WHERE codeUnit.incomingDependencies                        IS NOT NULL
      AND codeUnit.outgoingDependencies                        IS NOT NULL
      AND codeUnit.communityLocalClusteringCoefficient         IS NOT NULL
   RETURN DISTINCT 
         codeUnit.fqn                                         AS codeUnitName
        ,codeUnit.name                                        AS shortCodeUnitName
        ,artifact.name                                        AS projectName
        ,codeUnit.incomingDependencies                        AS incomingDependencies
        ,codeUnit.outgoingDependencies                        AS outgoingDependencies
        ,codeUnit.communityLocalClusteringCoefficient         AS clusteringCoefficient
"""

java_package_clustering_coefficient_features = query_cypher_to_data_frame(java_package_clustering_coefficient_query)
display(java_package_clustering_coefficient_features.head(5))

In [ ]:
# Visualize the distribution of clustering coefficients

def plot_clustering_coefficient_distribution(clustering_coefficients: pd.Series) -> None:
    """
    Plots the distribution of clustering coefficients.
    
    Parameters
    ----------
    clustering_coefficients : pd.Series
        Series containing clustering coefficient values.
    """
    if clustering_coefficients.empty:
        print("No data available to plot.")
        return

    plot.figure(figsize=(10, 6))
    plot.figure(figsize=(10, 6))
    plot.hist(clustering_coefficients, bins=40, color='blue', alpha=0.7, edgecolor='black')
    plot.title('Distribution of Clustering Coefficients')
    plot.xlabel('Clustering Coefficient')
    plot.ylabel('Frequency')
    plot.xlim(left=clustering_coefficients.min(), right=clustering_coefficients.max())
    # plot.yscale('log')  # Use logarithmic scale for better visibility of differences
    plot.grid(True)
    plot.tight_layout()

    # Vertical line for the mean
    mean_difference = clustering_coefficients.mean()
    plot.axvline(mean_difference, color='red', linestyle='dashed', linewidth=1, label=f'Mean: {mean_difference:.2f}')
    plot.legend()

    # Vertical line for 1 x standard deviations + mean (=z-score of 1)
    standard_deviation = clustering_coefficients.std()
    negative_z_score_1 = mean_difference - 1 * standard_deviation
    plot.axvline(negative_z_score_1, color='green', linestyle='dashed', linewidth=1, label=f'Mean + 1 x Standard Deviation (z-score 1): {negative_z_score_1:.2f}')
    plot.legend()

    plot.show()

In [ ]:
plot_clustering_coefficient_distribution(java_package_clustering_coefficient_features['clusteringCoefficient'])

## 2. Java Types

### 2.1 Differences between Page Rand and Article Rank


In [ ]:
java_type_anomaly_detection_centrality_features_query = """
    MATCH (artifact:Java:Artifact)-[:CONTAINS]->(codeUnit:Java:Type)
    WHERE codeUnit.incomingDependencies                        IS NOT NULL
      AND codeUnit.outgoingDependencies                        IS NOT NULL
      AND codeUnit.centralityArticleRank                       IS NOT NULL
      AND codeUnit.centralityPageRank                          IS NOT NULL
      AND codeUnit.centralityBetweenness                       IS NOT NULL
   RETURN DISTINCT 
         codeUnit.fqn                                         AS codeUnitName
        ,codeUnit.name                                        AS shortCodeUnitName
        ,artifact.name                                        AS projectName
        ,codeUnit.incomingDependencies                        AS incomingDependencies
        ,codeUnit.outgoingDependencies                        AS outgoingDependencies
        ,codeUnit.centralityArticleRank                       AS articleRank
        ,codeUnit.centralityPageRank                          AS pageRank
        ,codeUnit.centralityBetweenness                       AS betweenness
"""

java_type_anomaly_detection_centrality_features = query_cypher_to_data_frame(java_type_anomaly_detection_centrality_features_query)
display(java_type_anomaly_detection_centrality_features.head(5))

In [ ]:
plot_difference_between_article_and_page_rank(
    java_type_anomaly_detection_centrality_features['pageRank'],
    java_type_anomaly_detection_centrality_features['articleRank'],
    java_type_anomaly_detection_centrality_features['shortCodeUnitName']
)

### 2.2 Local Clustering Coefficient

In [ ]:
java_type_clustering_coefficient_query = """
    MATCH (artifact:Java:Artifact)-[:CONTAINS]->(codeUnit:Java:Type)
    WHERE codeUnit.incomingDependencies                        IS NOT NULL
      AND codeUnit.outgoingDependencies                        IS NOT NULL
      AND codeUnit.communityLocalClusteringCoefficient         IS NOT NULL
   RETURN DISTINCT 
         codeUnit.fqn                                         AS codeUnitName
        ,codeUnit.name                                        AS shortCodeUnitName
        ,artifact.name                                        AS projectName
        ,codeUnit.incomingDependencies                        AS incomingDependencies
        ,codeUnit.outgoingDependencies                        AS outgoingDependencies
        ,codeUnit.communityLocalClusteringCoefficient         AS clusteringCoefficient
"""

java_type_clustering_coefficient_features = query_cypher_to_data_frame(java_type_clustering_coefficient_query)
display(java_type_clustering_coefficient_features.head(5))

In [ ]:
plot_clustering_coefficient_distribution(java_type_clustering_coefficient_features['clusteringCoefficient'])